In [ ]:
%matplotlib inline

import numpy as np
import scanpy as sc
import torch
import pandas as pd
import scipy
import sys
sys.path.append("/home/anirudhn/Krushna/Thymus/Trajectory-Pos/Margaret/Margaret-main/margaret")
random_seed = 0
np.random.seed(random_seed)
torch.manual_seed(random_seed)

import matplotlib.pyplot as plt
rc_parms = {"figure.figsize": [4, 4], "figure.dpi": 300, "font.size": 10, "font.family": "Arial"}
save_parms = {"bbox_inches": "tight", "transparent": True}

In [ ]:
# Load data
data_path = "/home/anirudhn/Krushna/Thymus/hvae/outputs/Thymus_HierarVi.h5ad"
adata_all = sc.read_h5ad(data_path)
adata_all

In [ ]:
filtered_df = pd.DataFrame(data=adata_all.obsm['RNA_Z1_denoised'], index=adata_all.obs_names, columns=adata_all.var_names)
data = sc.AnnData(filtered_df)
data.obs = adata_all.obs[["batch","annotations"]]
data.obsm['Zc_totalVI'] = adata_all.obsm['Zc_totalVI']
data.obsm['X_umap'] = adata_all.obsm['Zc_umap']

In [ ]:
with plt.rc_context(rc_parms):
    ax = sc.pl.umap(data, color=["annotations"], frameon= False, return_fig = True, title='')
    plt.savefig("./Results/Figures/All-emb.png", **save_parms)


In [ ]:
pos_cells = pd.read_csv("../../Data/Thymus_CITE-seq-v1.0.0/YosefLab-Thymus_CITE-seq-9c20db9/Pseudotime/pseudotime_slingshot_2020913.csv")
pos_cells = pos_cells.Barcode.values

In [ ]:
data = data[pos_cells,:].copy()
data.obs['annotations'] = data.obs['annotations'].astype(str)

In [ ]:
del data.uns['annotations_colors']

In [ ]:
data.obs['annotations'].value_counts()

In [ ]:
Posclusters = ["DP (Sig.)", "Immature CD4", "Immature CD8", "Mature CD4", "Mature CD8"]
data = data[data.obs['annotations'].isin(Posclusters),:].copy()
# negatively selected cells, Treg cells, gamma-delta-like cells, mature cycling cells,
# and outlier clusters of doublets, interferon signature cells, and CD8-transgenic-specific outlier cells
print(data)
data.obs['annotations'] = data.obs['annotations'].astype('category')
data.obs['annotations'].value_counts()

In [ ]:
data.obs_names.to_frame().to_csv('./Results/Pos_subset.csv', index=False, header=False)

In [ ]:
# Posclusters = ["DP (Sig.)", "Immature CD4", "Immature CD8", "Mature CD4", "Mature CD8", "Interferon sig.", "Neg. sel. (2)", "Treg"]
# data = data[data.obs['annotations_clean'].isin(Posclusters),:].copy()
# data

In [ ]:
data

In [ ]:
with plt.rc_context(rc_parms):
    sc.pl.umap(data, color="annotations", frameon= False, return_fig = True, title='')
    plt.savefig("./Results/Figures/Postive-sel.png", **save_parms)


In [ ]:
sc.pp.neighbors(data, use_rep='Zc_totalVI')

In [ ]:
for res in [0.1,0.2,0.3,0.4,0.5,0.7,1]:
    print(res)
    sc.tl.leiden(data, resolution=res)
    sc.pl.umap(data, color = ['annotations', 'leiden'])

In [ ]:
data

In [ ]:
import warnings
from train_metric import train_metric_learner

with warnings.catch_warnings():
    # Filter out user warnings from PyTorch about saving scheduler state
    warnings.simplefilter("ignore")
    train_metric_learner(data, n_episodes=5, n_metric_epochs=30, obsm_data_key='Zc_totalVI', code_size=10,
        backend='leiden', init = "annotations", device='cuda', save_path='./Results/Thymus',
        cluster_kwargs={'random_state': 0, 'resolution': 0.4}, nn_kwargs={'random_state': 0, 'n_neighbors': 50},
        trainer_kwargs={'optimizer': 'SGD', 'lr': 0.01, 'batch_size': 256}
    )

In [ ]:
data.uns['metric_clustering_scores'] = list(map(str,data.uns['metric_clustering_scores']))

In [ ]:
X_embedded = generate_plot_embeddings(data.obsm['metric_embedding'], method='umap', random_state=random_seed) #preprocessed_data
data.obsm['X_met_embedding'] = X_embedded #preprocessed_data
data.obs['metric_clusters'] = data.obs['metric_clusters'].astype('category')

In [ ]:
# data.write("pos-margaret-annotation.h5ad")

In [ ]:
import numpy as np
from models.ti.connectivity import compute_directed_cluster_connectivity, compute_undirected_cluster_connectivity
from models.ti.graph import compute_trajectory_graph, compute_connectivity_graph
from utils.plot import plot_connectivity_graph, plot_trajectory_graph
from utils.util import get_start_cell_cluster_id

import networkx as nx

from sklearn.neighbors import NearestNeighbors
from models.ti.pseudotime import compute_pseudotime
from models.ti.pseudotime_v2 import compute_pseudotime
from models.ti.graph import compute_trajectory_graph_v2
from utils.plot import plot_trajectory_graph_v2
from utils.plot import plot_pseudotime

from utils.plot import generate_plot_embeddings, plot_gene_expression, plot_embeddings, plot_clusters
import matplotlib.pyplot as plt



In [ ]:
data = sc.read_h5ad("pos-margaret-annotation.h5ad")

In [ ]:
X = data.obsm['metric_embedding']

n_neighbors = 30
nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean").fit(X)
adj_dist = nbrs.kneighbors_graph(X, mode="distance")
adj_conn = nbrs.kneighbors_graph(X)

In [ ]:
with plt.rc_context(rc_parms):
    sc.pl.embedding(data, basis = 'X_met_embedding', color=['annotations', 'metric_clusters'],frameon=False,
                    legend_loc= 'on data', return_fig = True)
    plt.savefig("./Results/Figures/Margaret-pos.png", **save_parms)

In [ ]:
with plt.rc_context(rc_parms):
    sc.pl.umap(data, color = ['metric_clusters'], frameon=False, return_fig=True)
    plt.savefig("./Results/FiguresPos-emb-margaret-clusters.png", **save_parms)
    sc.pl.umap(data, color = ['annotations'], frameon=False, return_fig=True)
    plt.savefig("./Results/Figures/Pos-reumap-annotation.png", **save_parms)

In [ ]:
data.obs.groupby(['annotations','metric_clusters']).size().unstack().T

In [ ]:
communities = data.obs['metric_clusters'].to_numpy().astype(np.int32)
emb = pd.DataFrame(data.obsm['X_met_embedding'], index=data.obs_names)

# selecting the 10 cells based on the umap visualizatoin
emb = emb.loc[data.obs['metric_clusters'] == 1,:]
start_cell_ids = emb.iloc[np.argpartition(emb[1],10)[:10],:].index.to_list()
# emb.shape
start_cluster_ids = get_start_cell_cluster_id(data, start_cell_ids, communities)
# print(type(start_cluster_ids))
print(f'start cell ids {start_cluster_ids}')

In [ ]:
un_connectivity, un_z_score = compute_undirected_cluster_connectivity(communities, adj_conn, z_threshold=2.2)

In [ ]:
plot_connectivity_graph(data.obsm['X_met_embedding'], communities, un_connectivity, mode='undirected', offset=0.2, cmap='Blues', node_size=750)

In [ ]:
# connectivity, z_score = compute_directed_cluster_connectivity(communities, adj_conn, threshold=3.5)
# plot_connectivity_graph(data.obsm['X_met_embedding'], communities, connectivity, mode='undirected', offset=0.2, cmap='Blues', node_size=750)

In [ ]:
# v2 pseudotime
G_undirected, node_positions = compute_connectivity_graph(data.obsm['X_met_embedding'], data.obs['metric_clusters'], un_connectivity)
adj_cluster = nx.to_pandas_adjacency(G_undirected)

In [ ]:
G = nx.from_pandas_adjacency(adj_cluster)

# Draw the graph
pos = node_positions  # Positions for all nodes
nx.draw(G, pos, with_labels=True, labels={node: node for node in G.nodes()})#, node_size=50, node_color="skyblue", font_size=10, font_color="black", font_weight="bold")


In [ ]:
pseudotime = compute_pseudotime(data, start_cell_ids, adj_dist, adj_cluster)

In [ ]:
pseudotime = data.obs['metric_pseudotime_v2']

In [ ]:
# data.write("./pos-margaret-annotation.h5ad")

In [ ]:
# data.obsm['metric_branch_probs']= data.obsm['metric_branch_probs'].astype(str)

In [ ]:
# del data.obsm['metric_branch_probs']

In [ ]:
with plt.rc_context(rc_parms):
    sc.pl.embedding(data,'X_met_embedding', color = 'metric_pseudotime_v2', cmap='plasma', frameon=False, return_fig=True, title = '')
    plt.savefig("./Results/Figures/pseudotime.png", **save_parms)

In [ ]:
# Compute directed graph v2
# plot_trajectory_graph_v2(pseudotime, adj_cluster, data.obs['metric_clusters'], connectivity, node_positions, offset=0.2,node_size=2000, font_size = 20)

In [ ]:
def compute_trajectory_graph_v2(
    pseudotime, adj_cluster, communities, d_connectivity, norm=False
):
    n_communities = np.unique(communities).shape[0]
    cluster_ids = np.unique(communities)

    adj = pd.DataFrame(
        np.zeros((n_communities, n_communities)), index=cluster_ids, columns=cluster_ids
    )

    # Create cluster index
    cluster_pt = pd.DataFrame(index=cluster_ids)
    for idx in cluster_ids:
        cluster_idx = communities == idx
        cluster_pt.loc[idx, "t"] = np.mean(pseudotime.loc[cluster_idx])
    cols = adj_cluster.columns
    for idx in cluster_ids:
        connected_c_idx = cols[adj_cluster.loc[idx, :] != 0]
        for c_idx in connected_c_idx:
            if (cluster_pt.loc[c_idx, "t"] > cluster_pt.loc[idx, "t"]) and (
                adj_cluster.loc[c_idx, idx] != 0
            ):
                # The edge weight will be the contribution from the directed
                # connectivities and difference of the pseudotimes
                adj.loc[idx, c_idx] = d_connectivity.loc[idx, c_idx] + 1 / (
                    1 + np.exp(cluster_pt.loc[c_idx, "t"] - cluster_pt.loc[idx, "t"])
                )

    # Normalize the directed adjacency matrix
    if norm:
        adj = adj.div(adj.sum(axis=1), axis=0).fillna(0)
    return adj

# with cell type pie chats
def plot_trajectory_graph_v3(
    pseudotime,
    adj_cluster,
    communities,
    d_connectivity,
    node_positions,
    adata,
    cell_type = 'annotations',
    start_cell_ids=None,
    cmap="YlGn",
    figsize=(4,4),
    node_size=300,
    font_color="black",
    title=None,
    start_node_color=None,
    node_color=None,
    save_path=None,
    save_kwargs={},
    offset=0,
    **kwargs,
):
    adj_g = compute_trajectory_graph_v2(
        pseudotime, adj_cluster, communities, d_connectivity
    )
    g = nx.from_pandas_adjacency(adj_g, create_using=nx.DiGraph)

    if start_cell_ids is not None:
        start_cell_ids = (
            start_cell_ids if isinstance(start_cell_ids, list) else [start_cell_ids]
        )
    else:
        start_cell_ids = []

    start_cluster_ids = set([communities.loc[id] for id in start_cell_ids])

    colors = np.unique(communities)
    if node_color is not None:
        colors = []
        for c_id in np.unique(communities):
            if c_id in start_cluster_ids and start_node_color is not None:
                colors.append(start_node_color)
            else:
                colors.append(node_color)

    # Draw the graph
    fig = plt.figure(figsize=figsize)
    ax = plt.axes([0,0,1,1])
    
    if title is not None:
        plt.title(title)
    plt.axis("off")
    
    edge_weights = [offset + w for _, _, w in g.edges.data("weight")]
    
    nx.draw_networkx(
        g,
        pos=node_positions,
        cmap=cmap,
        node_color=colors,
        font_color=font_color,
        node_size=node_size,
        width=edge_weights,
        **kwargs,
    )
    
    trans = ax.transData.transform
    trans2 = fig.transFigure.inverted().transform

    piesize = 0.1##node_size*0.027/800#800->0.05 
    p2 = piesize/2.0
    # cs = cm.Set1(np.arange(15)/15.)
    
    for n in g:
        xx,yy = trans(node_positions[n]) # figure coordinates
        xa,ya = trans2((xx,yy)) # axes coordinates
        a = plt.axes([xa-p2,ya-p2, piesize, piesize])
        plt.title(n,**kwargs)
        a.set_aspect('equal')
        
        adata_node = adata[adata.obs['metric_clusters'] == n].copy()
        keys = adata.obs[cell_type].value_counts().keys()
        fracs = []
        colour_list = []
        color_map = adata.uns[cell_type+'_colors']
        for i,key in enumerate(sorted(keys)):
            fracs_keys = adata_node.obs[cell_type].value_counts().keys() 
            colour_list += [color_map[i]]
            if key in fracs_keys:
                fracs += [adata_node.obs[cell_type].value_counts()[key]]
            else:
                fracs += [0]
                
        fracs = np.array(fracs)/adata_node.shape[0] #[15,30,35, 10, 10]
        a.pie(fracs, colors = colour_list) # labels = keys
        
    plt.legend(sorted(keys),bbox_to_anchor = (9,5)) # loc = "lower left", 
    
    if save_path is not None:
        plt.savefig(save_path, **save_kwargs)
with plt.rc_context(rc_parms):
    plot_trajectory_graph_v3(pseudotime, adj_cluster, data.obs['metric_clusters'], un_connectivity, 
                             node_positions, offset=0.1, adata = data,node_size=500,
                            save_path = "./Results/Figures/Trajectory.png", save_kwargs = save_parms)#, fontsize = 20)

In [ ]:
from models.ti.downstream import (
    get_terminal_states,
    get_terminal_cells,
    sample_waypoints,
    compute_diff_potential
)

In [ ]:
import sys, importlib
importlib.reload(sys.modules['models.ti.downstream'])


In [ ]:
# del get_terminal_states
# del get_terminal_cells
# del sample_waypoints
# del compute_diff_potential

In [ ]:
data

In [ ]:
# G_directed_v2 = compute_trajectory_graph_v2(pseudotime, adj_cluster, data.obs['metric_clusters'], connectivity)
# terminal_clusters = get_terminal_states(data, G_directed_v2, start_cell_ids, mad_multiplier=5)
terminal_clusters = {2,8}
data.uns["metric_terminal_clusters"] = list(terminal_clusters)
t_cell_ids = get_terminal_cells(data)
_ = sample_waypoints(data, adj_dist.todense(), n_waypoints=30) #dists, wp

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as ss

from numpy.linalg import inv, pinv
from numpy.linalg.linalg import LinAlgError
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import find, csr_matrix
from scipy.stats import entropy
from scipy.sparse.csgraph import dijkstra

from models.ti.sim import compute_lpi, compute_lrw
from utils.util import get_start_cell_cluster_id, prune_network_edges

def draw_network(embeddings, adjacency_matrix, node_colors='blue'):
    G= nx.from_numpy_array(adjacency_matrix)

    plt.figure(figsize=(8, 6))
    pos = {i: (embeddings[i, 0], embeddings[i, 1]) for i in range(len(embeddings))}
    
    # Draw the nodes
    nx.draw_networkx_nodes(G, pos, node_size=3, node_color=node_colors)
    
    # Draw the edges
    nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5, edge_color='gray')
    plt.show()

def prune_network_edges(communities, adj_sc, adj_cluster):
    G = nx.from_pandas_adjacency(adj_cluster)
    pos = node_positions 
    nx.draw(G, pos, with_labels=True, labels={node: node for node in G.nodes()})
    plt.show()
    
    cluster_ids = np.unique(communities)

    # Create cluster index
    clusters = {}
    for idx in cluster_ids:
        cluster_idx = communities == idx
        clusters[idx] = cluster_idx

    col_ids = adj_cluster.columns

    for c_idx in adj_cluster.index:
        cluster_i = clusters[c_idx]
        non_connected_clusters = col_ids[adj_cluster.loc[c_idx, :] == 0]
        for nc_idx in non_connected_clusters:
            if nc_idx == c_idx:
                continue
            cluster_nc = clusters[nc_idx]

            # Prune (remove the edges between two non-connected clusters)
            adj_sc.loc[cluster_i, cluster_nc] = 0

    return adj_sc

def series_to_colors(values):
    import seaborn as sns
    import matplotlib.colors as mcolors
    
    unique_categories = list(set(values))
    palette = sns.color_palette("hsv", len(unique_categories))
    color_mapping = {category: mcolors.rgb2hex(palette[i]) for i, category in enumerate(unique_categories)}
    colors = values.map(color_mapping)
    return colors

def _construct_markov_chain(
    wp_data, #only waypoint cells
    knn,
    pseudotime,
    comms,
    adj_cluster,
    std_factor=1.0,
    prune_wp_graph=True,
    n_jobs=1,
    embeddings_df = None,
    terminal_states = None,
):

    # Markov chain construction
    print("Markov chain construction...")
    waypoints = wp_data.index

    # kNN graph
    n_neighbors = knn
    nbrs = NearestNeighbors(
        n_neighbors=n_neighbors, metric="euclidean", n_jobs=n_jobs
    ).fit(wp_data)
    kNN = nbrs.kneighbors_graph(wp_data, mode="distance")
    dist, ind = nbrs.kneighbors(wp_data)

    # Prune the kNN graph wrt to the undirected graph
    if prune_wp_graph:
        wp = wp_data.index
        kNN_pruned = pd.DataFrame(kNN.todense(), index=wp, columns=wp)
        
        embeddings_df = embeddings_df.loc[wp,:]
        # draw_network(embeddings_df.values, kNN_pruned.to_numpy(), node_colors=series_to_colors(comms).values)
        
        kNN_pruned = prune_network_edges(comms.loc[wp], kNN_pruned, adj_cluster)
        kNN = csr_matrix(kNN_pruned)
        embeddings_df = embeddings_df.loc[wp,:]
        # print('kNN pruned')
        # draw_network(embeddings_df.values, kNN_pruned.to_numpy(),node_colors=series_to_colors(comms).values)

    # Standard deviation allowing for "back" edges
    adpative_k = np.min([int(np.floor(n_neighbors / 3)) - 1, 30])
    adaptive_std = np.ravel(dist[:, adpative_k])

    # Directed graph construction
    # pseudotime position of all the neighbors
    traj_nbrs = pd.DataFrame(
        pseudotime[np.ravel(waypoints.values[ind])].values.reshape(
            [len(waypoints), n_neighbors]
        ),
        index=waypoints,
    )

    # Remove edges that move backwards in pseudotime except for edges that are within
    # the computed standard deviation
    rem_edges = traj_nbrs.apply(
        lambda x: x < pseudotime[traj_nbrs.index] - std_factor * adaptive_std
    )
    rem_edges = rem_edges.stack()[rem_edges.stack()]
    print('remove edges')
    print(rem_edges)
    # Determine the indices and update adjacency matrix
    cell_mapping = pd.Series(range(len(waypoints)), index=waypoints)
    x = list(cell_mapping[rem_edges.index.get_level_values(0)])
    y = list(rem_edges.index.get_level_values(1))

    print(type(kNN), 'knn type')
   
    # Update adjacecy matrix
    kNN[x, ind[x, y]] = 0
    print("updated kNN")
    draw_network(embeddings_df.values, kNN.todense(), node_colors=series_to_colors(comms).values)

    
    print('plotting edges that are removed')
    kNN_temp = np.zeros(kNN.shape)
    kNN_temp[x, ind[x, y]] = 1
    draw_network(embeddings_df.values, kNN_temp, node_colors=series_to_colors(comms).values)

    
    # removing the nodes with no outgoing edges from adj matrix
    # and they should not be terminal states
    no_edge_ind = np.where(kNN.sum(axis=1)==0)[0]
    terminal_ind = np.where(waypoints.isin(terminal_states))[0]

    
    print('allowed zeros in knn',set(no_edge_ind) & set(terminal_ind))
    no_edge_ind = np.array(list(set(no_edge_ind) - set(terminal_ind)))
    no_edge_cells = np.array(list(wp_data.iloc[no_edge_ind,:].index))
    
    def add_non_back_edge(wp_data, no_edge_ind, pseudotime, std_factor, comms, adj_cluster):
        # @wp_data data frame containing all waypoints
        # @no_edge_ind index of cells for which we need to add an edge
        # @pseudotime series object
        # @std_factor pa
        # @comm series of communies with indexed cell names to not to have pruned edge
        nbrs = NearestNeighbors(
                n_neighbors=wp_data.shape[0], metric="euclidean", n_jobs=n_jobs
            ).fit(wp_data)
        
        wp_data_query = wp_data.iloc[no_edge_ind,:]
        wp_data_query = wp_data_query.index
        print('waypoints len', len(waypoints))
        dist, ind = nbrs.kneighbors(wp_data.iloc[no_edge_ind,:].values, wp_data.shape[0], return_distance=True)
        traj_nbrs = pd.DataFrame(
            pseudotime[np.ravel(wp_data.index.values[ind])].values.reshape(
                [len(wp_data_query), wp_data.shape[0]]
            ),
            index=wp_data_query,
        )
        n_neighbors = wp_data.shape[0]
        adpative_k = np.min([int(np.floor(n_neighbors / 3)) - 1, 30])
        adaptive_std = np.ravel(dist[:, adpative_k])
        rem_edges = traj_nbrs.apply(
            lambda x: x < pseudotime[traj_nbrs.index] - std_factor * adaptive_std
        )
        add_edges_ij = []
        dist_ij = []

        for i in range(rem_edges.shape[0]):
            for j in range(1,rem_edges.shape[1]): # removing the first neighbor as it is self
                if not rem_edges.iloc[i,j]:
                    cell_i = rem_edges.index[i]
                    cell_j = wp_data.index[ind[i,j]]
                    if comms[cell_i]==comms[cell_i] or adj_cluster.loc[comms[cell_i],comms[cell_i]] != 0 :
                        add_edges_ij.append([cell_i,cell_j])
                        dist_ij.append(dist[i,j])
                        break
            else:
                raise NotImplementedError(
                    'There is no neighbour point having higher sudotime\n\
                    tyring adjusting std_factor or no_ofwaypoints'
                )
        
        add_edges_ij = np.array(add_edges_ij)
        dist_ij = np.array(dist_ij)
        
        print('Adding edges ', add_edges_ij)
        # xy = wp_data.index[add_edges_ij.ravel()].reshape(add_edges_ij.shape)
        x,y,dist = add_edges_ij[:,0], add_edges_ij[:,1], dist_ij
        return x,y,dist
    add_x, add_y, dist_ij = add_non_back_edge(wp_data, no_edge_ind, pseudotime, std_factor, comms, adj_cluster)
    add_x, add_y = cell_mapping[add_x], cell_mapping[add_y]
    kNN[add_x,add_y] = dist_ij
    
    x, y, z = find(kNN)
    aff = np.exp(
        -(z ** 2) / (adaptive_std[x] ** 2) * 0.5
        - (z ** 2) / (adaptive_std[y] ** 2) * 0.5
    )
    W = csr_matrix((aff, (x, y)), [len(waypoints), len(waypoints)])

    # Transition matrix
    D = np.ravel(W.sum(axis=1))
    x, y, z = find(W)
    T = csr_matrix((z / D[x], (x, y)), [len(waypoints), len(waypoints)])
    print('no of 0 trans prob @return', np.sum(T.sum(axis=1)<0.8))

    

    return T, no_edge_cells

def _differentiation_entropy(
    wp_data,
    terminal_states,
    knn,
    pseudotime,
    comms,
    adj_cluster,
    std_factor=1.0,
    n_jobs=1,
    prune_wp_graph=True,
    embeddings_df = None,
):

    T,no_edge_cells = _construct_markov_chain(
        wp_data,
        knn,
        pseudotime,
        comms,
        adj_cluster,
        std_factor=std_factor,
        n_jobs=n_jobs,
        prune_wp_graph=prune_wp_graph,
        embeddings_df = embeddings_df,
        terminal_states = terminal_states,
    )
    wp_data = wp_data.drop(no_edge_cells)
    # Absorption states should not have outgoing edges
    waypoints = wp_data.index
    print('waypoints len', len(waypoints))
    abs_states = waypoints.isin(terminal_states)
    abs_states = np.where(abs_states)[0]
    print('abs states',abs_states)
    # Reset absorption state affinities by Removing neigbors
    T[abs_states, :] = 0
    # Diagnoals as 1s
    T[abs_states, abs_states] = 1
    print('zeros in tranprob matrix', sum(T.sum(axis=1)<0.9))

    # Fundamental matrix and absorption probabilities
    print("Computing fundamental matrix and absorption probabilities...")
    # Transition states
    trans_states = list(set(range(len(waypoints))).difference(abs_states))

    # Q matrix
    Q = T[trans_states, :][:, trans_states]
    # Fundamental matrix
    mat = np.eye(Q.shape[0]) - Q.todense()
    try:
        N = inv(mat)
    except LinAlgError:
        # Compute the pseudoinverse if the main inv cannot be computed
        N = pinv(mat)

    # Absorption probabilities
    print('shape of N', N.shape)
    branch_probs = np.dot(N, T[trans_states, :][:, abs_states].todense())
    branch_probs = pd.DataFrame(
        branch_probs, index=waypoints[trans_states], columns=waypoints[abs_states]
    )
    branch_probs[branch_probs < 0] = 0

    # Entropy
    ent = branch_probs.apply(entropy, axis=1)

    # Add terminal states
    ent = pd.concat([ent,pd.Series(0, index=terminal_states)])
    bp = pd.DataFrame(0, index=terminal_states, columns=terminal_states)
    bp.values[range(len(terminal_states)), range(len(terminal_states))] = 1
    branch_probs = pd.concat([branch_probs,bp.loc[:, branch_probs.columns]])

    return ent, branch_probs
    
def compute_diff_potential(
    ad,
    adj_dist,
    adj_cluster,
    comms_key="metric_clusters",
    embed_key="metric_embedding",
    pt_key="metric_pseudotime_v2",
    wp_key="metric_waypoints",
    tc_key="metric_terminal_cells",
    sim_scheme="lrw",
    sim_kwargs={},
    std_factor=1.0,
    knn=30,
    prune_wp_graph=True,
    exclude_clusters=None,
    n_jobs=1,
    embeddings = None
):
    wps = ad.uns[wp_key]
    if  type(data.uns['metric_waypoints']) == np.ndarray:
        wps = wps.tolist()
    # Add the terminal cells to the wp list
    t_cell_ids = ad.uns[tc_key]
    wps.extend(t_cell_ids)
    wp_ = set(wps)

    # Cell to Waypoint Connectivity
    print("Cell to Waypoint connectivity")
    communities = ad.obs[comms_key]
    adj_dist_pruned = prune_network_edges(
        communities,
        pd.DataFrame(adj_dist, index=ad.obs_names, columns=ad.obs_names),
        adj_cluster,
    )
    print("adj_dist_pruned")
    # draw_network(embeddings, adj_dist_pruned.to_numpy())

    adj_dist_pruned[adj_dist_pruned == 0] = np.inf


    if exclude_clusters is not None:
        for cid in exclude_clusters:
            idx = ad.obs_names[communities == cid]
            adj_dist_pruned = adj_dist_pruned.drop(index=idx, columns=idx)
            adj_cluster = adj_cluster.drop(index=cid, columns=cid)

    # This represents the final connectivity Adj mat. between cells
    adj_conn = csr_matrix(np.exp(-np.power(adj_dist_pruned, 2)))

    if sim_scheme == "lpi":
        S = compute_lpi(adj_conn, **sim_kwargs)
    elif sim_scheme == "lrw":
        S = compute_lrw(adj_conn, **sim_kwargs)
    else:
        raise NotImplementedError(
            f"The scheme {sim_scheme} has not been implemented yet!"
        )

    S = pd.DataFrame(
        S.todense(), index=adj_dist_pruned.index, columns=adj_dist_pruned.index
    )

    # Waypoint to Terminal State connectivity
    print("Waypoint to Terminal State connectivity")
    X = pd.DataFrame(ad.obsm[embed_key], index=ad.obs_names)
    X_wp = X.loc[list(wp_), :] # only waypoints
    pt = ad.obs[pt_key]
    wp_comms = communities.loc[list(wp_)]
    _, bp = _differentiation_entropy(
        X_wp,
        t_cell_ids,
        knn,
        pt,
        wp_comms,
        adj_cluster,
        std_factor=std_factor,
        prune_wp_graph=prune_wp_graph,
        n_jobs=n_jobs,
        embeddings_df = pd.DataFrame(embeddings, index=ad.obs_names),
       
    )

    # Project branch probs on the cells
    bps_ = S.loc[:, list(wp_)].loc[:, bp.index].dot(bp)
    bps_ = bps_.div(bps_.sum(axis=1), axis=0)

    # We perform the assignment in this way to account for
    # clusters that might have been excluded.
    bps = pd.DataFrame(index=ad.obs_names, columns=bps_.columns)
    bps.loc[bps_.index, :] = bps_

    # Compute row-wise entropy to compute DP
    ent_ = ss.entropy(bps_, base=2, axis=1)
    ent = pd.Series(index=ad.obs_names)
    ent.loc[adj_dist_pruned.index] = ent_

    # Add to the anndata object
    ad.obsm["metric_branch_probs"] = bps
    ad.obs["metric_dp"] = ent

    return ent, bps

In [ ]:
ent, bps = compute_diff_potential(data, adj_dist.todense(), adj_cluster, std_factor=1,
                                  prune_wp_graph=True, embeddings = data.obsm['X_met_embedding'])

# Plot the Differentiation potential
plot_embeddings(
    data.obsm['X_met_embedding'],
    s=1,
    c=ent,
    figsize=(8, 8),
    cmap='plasma',
    show_colorbar=True,
    cb_axes_pos=[0.92, 0.55, 0.02, 0.3],
    save_kwargs={
        'dpi': 300,
        'bbox_inches': 'tight',
        'transparent': True
    }
)

In [ ]:
ts_map = {
    8: "Mature CD8",
    2: "Mature CD4",
    'Mature CD8': 'Mature CD8',
    "Mature CD4": "Mature CD4"
}
color_map = {
    2: '#9467bd',
    8: '#d62728',
    "Mature CD8": '#9467bd',
    'Mature CD4': '#d62728',
}

In [ ]:
for pro in ["CD24","CD62L","CD55"]:
    match =  adata_all.obs.columns[adata_all.obs.columns.to_series().str.contains(pro)]
    data.obs[pro] = adata_all.obs.loc[data.obs_names, match.values[0]]

In [ ]:
# Plot lineage trends
import importlib
importlib.reload(sys.modules['utils.plot'])
from utils.plot import plot_lineage_trends


comms = data.obs['metric_clusters'].loc[bps.columns]
bps_ = pd.DataFrame(bps.to_numpy(), columns=list(comms), index=data.obs_names)

In [ ]:
bps_.rename(columns=ts_map).to_csv("./branch_probabilities.csv")

In [ ]:
bps_ = pd.read_csv("./branch_probabilities.csv", index_col=0)
bps_

In [ ]:
bps_["Mature CD4"].hist()

In [ ]:
importlib.reload(sys.modules['utils.plot'])
from utils.plot import plot_lineage_trends 


In [ ]:
with plt.rc_context(rc_parms):
    
    for gene in ['Rag1', 'Cxcr4', 'Trbc1' , 'Ccr9', 'Ccr4', 'Ccr7', 'H2-K1', 'Klf2', 'S1pr1', "CD24","CD62L","CD55"]:
        plot_lineage_trends(
            data,
            bps_,
            [gene],
            pseudotime_key='metric_pseudotime_v2',
            figsize=(2,2),
            # imputed_key='X_magic',
            nrows=1,
            norm=True,
            ts_map=ts_map,
            save_path='./Results/Figures/Trends/'+gene+'.png',
            save_kwargs={
                'dpi': 300,
                'bbox_inches': 'tight',
                'transparent': True
            },
            color_map=color_map,
            loc = None,
            show_title = False,
            set_ylabel = None
        )

In [ ]:
data_raw = data.copy()
data_raw.X = adata_all[data_raw.obs_names,:].X


In [ ]:
for pro in ["CD24","CD62L","CD55"]:
    match =  adata_all.obs.columns[adata_all.obs.columns.to_series().str.contains(pro)]
    data_raw.obs[pro] = adata_all.obs.loc[data_raw.obs_names, match.values[0]]


In [ ]:
# importlib.reload(sys.modules['utils.plot'])
from utils.plot import plot_lineage_trends

In [ ]:
for gene in ['Rag1', 'Cxcr4', 'Trbc1' , 'Ccr9', 'Ccr4', 'Ccr7', 'H2-K1', 'Klf2', 'S1pr1', "CD24","CD62L","CD55"]:
    print(gene)
    plot_lineage_trends(
        data_raw,
        bps_,
        [gene],
        pseudotime_key='metric_pseudotime_v2',
        figsize=(3,3),
        # imputed_key='X_magic',
        nrows=1,
        norm=True,
        ts_map=ts_map,
        show_title=True,
        # save_path='./lineage_1.png',
        save_kwargs={
            'dpi': 300,
            'bbox_inches': 'tight',
            'transparent': True
        },
        color_map=color_map,
        
        # threshold=0
    )

# recheck

In [ ]:
# from sklearn.neighbors import NearestNeighbors
# from models.ti.connectivity import compute_undirected_cluster_connectivity, compute_directed_cluster_connectivity

# from utils.plot import plot_connectivity_graph, plot_trajectory_graph, plot_trajectory_graph_v2, plot_embeddings
# from models.ti.graph import  compute_connectivity_graph
# import networkx as nx
# from models.ti.pseudotime_v2 import compute_pseudotime

# data = sc.read_h5ad("./pos-margaret-annotation.h5ad")

# communities = data.obs['metric_clusters'].to_numpy().astype(np.int32)
# X = data.obsm['metric_embedding']
# nbrs = NearestNeighbors(n_neighbors=30, metric="euclidean").fit(X)
# adj_dist = nbrs.kneighbors_graph(X, mode="distance")
# adj_conn = nbrs.kneighbors_graph(X)
# un_connectivity, un_z_score = compute_undirected_cluster_connectivity(communities, adj_conn, z_threshold=2.2)
# plot_connectivity_graph(data.obsm['X_met_embedding'], communities, un_connectivity, mode='undirected', figsize=(3,3), node_size=150)#, offset=0.2, cmap='Blues', node_size=750)

# G_undirected, node_positions = compute_connectivity_graph(data.obsm['X_met_embedding'], data.obs['metric_clusters'], un_connectivity)
# adj_cluster = nx.to_pandas_adjacency(G_undirected)

# start_cluster_ids = {1}
# start_cell_ids = data[data.obs['metric_clusters'].isin(list(start_cluster_ids)),:].obs_names.to_list()
# pseudotime = compute_pseudotime(data, start_cell_ids, adj_dist, adj_cluster)
# sc.pl.embedding(data,'X_met_embedding', color = 'metric_pseudotime_v2', cmap='plasma', frameon=False, return_fig=True, title='')


# from utils.plot import plot_lineage_trends


# from models.ti.downstream import  compute_diff_potential, sample_waypoints
# from models.ti.graph import compute_trajectory_graph_v2
# from models.ti.downstream import   get_terminal_states, get_terminal_cells
# connectivity, z_score = compute_directed_cluster_connectivity(communities, adj_conn, threshold=2)
# plot_trajectory_graph_v2(pseudotime, adj_cluster, data.obs['metric_clusters'], connectivity, node_positions, offset=0.2, figsize = (4,4))
# G_directed_v2 = compute_trajectory_graph_v2(pseudotime, adj_cluster, data.obs['metric_clusters'], connectivity)
# terminal_clusters = get_terminal_states(data, G_directed_v2, start_cell_ids)#, mad_multiplier=0.7)
# t_cell_ids = get_terminal_cells(data)
# _ = sample_waypoints(data, adj_dist.todense(), n_waypoints=30) #dists, wp
# ent, bps = compute_diff_potential(data, adj_dist.todense(), adj_cluster, prune_wp_graph=True, std_factor=0)

# # Plot the Differentiation potential
# plot_embeddings(data.obsm['X_met_embedding'],
#     s=1,
#     c=ent,
#     figsize=(6,6),
#     cmap='plasma',
#     show_colorbar=True,
# )